# Lección 09 - Patrón de Diseño de Metacognición


## Configuración

Este cuaderno demuestra el patrón de diseño de Metacognición utilizando el Microsoft Agent Framework.

**Requisitos previos:**
- Implementación de Azure OpenAI configurada mediante variables de entorno
- Azure CLI autenticado (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ¿Qué es la Metacognición?

La metacognición es **pensar sobre el pensamiento**. En el contexto de los agentes de IA, significa construir agentes que puedan:

- **Autorreflexionar** sobre sus propios resultados y proceso de razonamiento
- **Detectar errores** y recuperarse de manera elegante en lugar de fallar silenciosamente
- **Evaluar** si sus respuestas son completas y útiles
- **Adaptar** su estrategia cuando un enfoque inicial no funciona (por ejemplo, recurrir a un sistema de respaldo)

Un agente metacognitivo no solo responde preguntas — monitorea su propio rendimiento y se ajusta sobre la marcha.


## Herramientas Primaria y de Respaldo

Un patrón común de metacognición es la **estrategia de reserva**. El agente primero intenta una herramienta primaria; si falla (por ejemplo, un error 404), el agente reconoce la falla y cambia transparentemente a una herramienta de respaldo.

Esto refleja sistemas del mundo real donde los servicios primarios pueden no estar disponibles y los agentes deben autodiagnosticar el problema antes de elegir una ruta alternativa.

A continuación definimos dos herramientas de búsqueda de vuelos:
- **Primaria** — cubre París, Tokio y Barcelona
- **Respaldo** — cubre Berlín, Sídney y Ciudad de Nueva York


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Agente de Auto-reflexión con Recuperación de Errores

El agente a continuación está instruido para probar primero el sistema de vuelo principal, reconocer fallos y recurrir de manera transparente al sistema de respaldo. Después de cada respuesta, se autoevalúa brevemente para determinar si respondió completamente a la pregunta del usuario.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Patrón de Autoevaluación

Otra faceta de la metacognición es la **autoevaluación**: un agente separado (o el mismo agente en una segunda pasada) revisa una respuesta para verificar su completitud, exactitud y utilidad.

A continuación, creamos un agente `ResponseEvaluator` que puntúa las respuestas de agentes de viajes en tres dimensiones.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Resumen

En esta lección aprendiste a construir **agentes metacognitivos** usando el Microsoft Agent Framework:

- **Autorreflexión**: Agentes que monitorean su propio razonamiento y comunican de manera transparente lo que sucedió.
- **Recuperación de errores con alternativas**: Un patrón de herramienta principal + de respaldo en donde el agente detecta fallos (p. ej., errores 404) y automáticamente intenta una fuente alternativa.
- **Autoevaluación**: Un agente evaluador separado que califica las respuestas en cuanto a completitud, precisión y utilidad.

Estos patrones hacen que los agentes sean más robustos, transparentes y confiables, cualidades críticas para implementaciones en producción.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por la precisión, tenga en cuenta que las traducciones automáticas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda la traducción profesional realizada por humanos. No nos hacemos responsables por malentendidos o interpretaciones erróneas que puedan derivarse del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
